In [1]:
from pathlib import Path

In [5]:
import os

In [13]:
import pandas as pd

In [19]:
import numpy as np

In [29]:
import json

In [34]:
from sklearn.preprocessing import StandardScaler

In [42]:
from sklearn.ensemble import IsolationForest

In [46]:
from sklearn.metrics import classification_report,confusion_matrix

In [49]:
from sklearn.neighbors import LocalOutlierFactor

In [50]:
from sklearn.svm import OneClassSVM

In [6]:
os.getcwd()

'/Users/zainabfirdaus/git/learn/python3/notebooks'

In [15]:
try :
    HERE = Path(__file__).parent
except NameError :
    HERE = Path( os.getcwd())

In [16]:

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_FILE   = HERE / "sensor_records.json"
OUTPUT_JSON = HERE / "anomaly_results.json"
OUTPUT_PLOT = HERE / "anomaly_plots.png"

# ── Features used for detection ───────────────────────────────────────────────
FEATURE_COLS = [
    "wind_speed_ms",
    "rotor_speed_rpm",
    "power_output_kw",
    "generator_temp_c",
    "vibration_axial_mms2",
    "vibration_radial_mms2",
    "yaw_angle_deg",
]

# Derived feature: power coefficient (should be ~constant for healthy turbine)
# Added after scaling to catch power-drop and yaw-misalignment subtly
DERIVED_COLS = ["power_per_wind_cube"]


In [17]:

# ── 1. Load data ──────────────────────────────────────────────────────────────
def load_data(path: Path) -> pd.DataFrame:
    with open(path) as f:
        raw = json.load(f)

    rows = []
    for r in raw["records"]:
        row = {
            "timestamp":        r["timestamp"],
            "turbine_id":       r["turbine_id"],
            "anomaly_true":     int(r["edge_status"]["anomaly_detected"]),
            "anomaly_type_true": r["edge_status"].get("anomaly_type"),
        }
        row.update(r["metrics"])
        rows.append(row)

    df = pd.DataFrame(rows)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)

    # Derived feature: power / wind³ (power coefficient proxy)
    wind_cube = df["wind_speed_ms"].clip(lower=0.1) ** 3
    df["power_per_wind_cube"] = df["power_output_kw"] / wind_cube

    return df, raw["metadata"]

In [20]:

# ── 2. Build feature matrix ───────────────────────────────────────────────────
def build_features(df: pd.DataFrame) -> np.ndarray:
    cols = FEATURE_COLS + DERIVED_COLS
    X = df[cols].values
    scaler = StandardScaler()
    return scaler.fit_transform(X), scaler, cols

In [21]:

# ── 3. Models ─────────────────────────────────────────────────────────────────
def run_isolation_forest(X: np.ndarray, contamination: float) -> np.ndarray:
    model = IsolationForest(
        n_estimators=200,
        contamination=contamination,
        random_state=42,
        n_jobs=-1,
    )
    preds = model.fit_predict(X)           # -1 = anomaly, 1 = normal
    scores = model.decision_function(X)    # lower = more anomalous
    return (preds == -1).astype(int), scores



In [22]:

def run_lof(X: np.ndarray, contamination: float) -> np.ndarray:
    model = LocalOutlierFactor(
        n_neighbors=20,
        contamination=contamination,
        novelty=False,
    )
    preds = model.fit_predict(X)
    scores = -model.negative_outlier_factor_  # higher = more anomalous
    return (preds == -1).astype(int), scores


In [23]:

def run_one_class_svm(X: np.ndarray, contamination: float) -> np.ndarray:
    # Train on the majority (~normal) subset using IsolationForest pre-filter
    pre = IsolationForest(contamination=contamination, random_state=42)
    pre.fit(X)
    normal_mask = pre.predict(X) == 1
    X_train = X[normal_mask]

    model = OneClassSVM(kernel="rbf", gamma="auto", nu=contamination)
    model.fit(X_train)
    preds = model.predict(X)
    scores = -model.decision_function(X)   # higher = more anomalous
    return (preds == -1).astype(int), scores


In [24]:

# ── 4. Evaluate ───────────────────────────────────────────────────────────────
def evaluate(name: str, y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    print(f"\n{'─'*60}")
    print(f"  {name}")
    print(f"{'─'*60}")
    print(classification_report(y_true, y_pred, target_names=["Normal", "Anomaly"],
                                 zero_division=0))
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    print(f"  Confusion matrix  TP={tp}  FP={fp}  FN={fn}  TN={tn}")

    report = classification_report(y_true, y_pred,
                                    target_names=["Normal", "Anomaly"],
                                    output_dict=True, zero_division=0)
    return {
        "precision": round(report["Anomaly"]["precision"], 3),
        "recall":    round(report["Anomaly"]["recall"], 3),
        "f1":        round(report["Anomaly"]["f1-score"], 3),
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
    }



In [25]:

# ── 5. Plot ───────────────────────────────────────────────────────────────────
def plot_results(df: pd.DataFrame, results: dict, out_path: Path):
    metrics_to_plot = [
        ("generator_temp_c",       "Generator Temp (°C)"),
        ("vibration_axial_mms2",   "Vibration Axial (mm/s²)"),
        ("power_output_kw",        "Power Output (kW)"),
        ("power_per_wind_cube",    "Power / Wind³ (proxy Cp)"),
    ]
    methods = list(results.keys())
    colors  = {"Isolation Forest": "#e74c3c",
               "LOF":              "#e67e22",
               "One-Class SVM":    "#9b59b6"}

    n_metrics = len(metrics_to_plot)
    fig, axes = plt.subplots(n_metrics, 1, figsize=(16, 4 * n_metrics), sharex=True)
    fig.suptitle("Wind Turbine Anomaly Detection — WT-402B", fontsize=14, fontweight="bold")

    t = df["timestamp"]

    for ax, (col, label) in zip(axes, metrics_to_plot):
        ax.plot(t, df[col], color="#2c3e50", linewidth=0.8, alpha=0.85, label="Sensor reading")

        # Ground truth shading
        gt_mask = df["anomaly_true"] == 1
        ax.fill_between(t, ax.get_ylim()[0], ax.get_ylim()[1],
                        where=gt_mask, alpha=0.08, color="green", label="Ground truth anomaly")

        # Per-method markers
        for method in methods:
            pred_mask = results[method]["pred"] == 1
            ax.scatter(t[pred_mask], df.loc[pred_mask, col],
                       color=colors[method], s=18, zorder=5,
                       alpha=0.7, label=f"{method} detection")

        ax.set_ylabel(label, fontsize=9)
        ax.grid(True, linestyle="--", alpha=0.4)

    # Legend on last axis
    patches = [mpatches.Patch(color="#2ecc71", alpha=0.4, label="Ground truth")]
    patches += [mpatches.Patch(color=colors[m], label=m) for m in methods]
    axes[-1].legend(handles=patches, loc="upper right", fontsize=8)
    axes[-1].set_xlabel("Timestamp", fontsize=9)

    plt.tight_layout()
    plt.savefig(out_path, dpi=130, bbox_inches="tight")
    print(f"\n  Plot saved → {out_path.name}")



In [26]:

# ── 6. Save results ───────────────────────────────────────────────────────────
def save_results(df: pd.DataFrame, results: dict, metadata: dict, out_path: Path):
    records_out = []
    for i, row in df.iterrows():
        records_out.append({
            "index":            i,
            "timestamp":        row["timestamp"].isoformat(),
            "anomaly_true":     bool(row["anomaly_true"]),
            "anomaly_type_true": row["anomaly_type_true"],
            "predictions": {
                method: {
                    "anomaly_predicted": bool(results[method]["pred"][i]),
                    "anomaly_score":     round(float(results[method]["score"][i]), 5),
                }
                for method in results
            },
        })

    out = {
        "metadata": metadata,
        "model_metrics": {m: results[m]["eval"] for m in results},
        "records": records_out,
    }
    with open(out_path, "w") as f:
        json.dump(out, f, indent=2, default=str)
    print(f"  Results saved → {out_path.name}")


### Loading Data

In [30]:
df, metadata = load_data(DATA_FILE)

In [31]:
print(f"  {len(df)} records | {df['anomaly_true'].sum()} ground-truth anomalies")


  500 records | 46 ground-truth anomalies


In [35]:
X, scaler, feat_cols = build_features(df)

In [36]:
y_true = df["anomaly_true"].values

In [37]:
contamination = round(df["anomaly_true"].mean(), 3)

In [38]:
print(f"  Contamination rate: {contamination:.1%}  |  Features: {feat_cols}")

  Contamination rate: 9.2%  |  Features: ['wind_speed_ms', 'rotor_speed_rpm', 'power_output_kw', 'generator_temp_c', 'vibration_axial_mms2', 'vibration_radial_mms2', 'yaw_angle_deg', 'power_per_wind_cube']


### Running Models

In [40]:
methods = {
        "Isolation Forest": run_isolation_forest,
        "LOF":              run_lof,
        "One-Class SVM":    run_one_class_svm,
}

results = {}

In [51]:
for name, fn in methods.items():
        pred, score = fn(X, contamination)
        metrics = evaluate(name, y_true, pred)
        results[name] = {"pred": pred, "score": score, "eval": metrics}


────────────────────────────────────────────────────────────
  Isolation Forest
────────────────────────────────────────────────────────────
              precision    recall  f1-score   support

      Normal       0.98      0.98      0.98       454
     Anomaly       0.80      0.80      0.80        46

    accuracy                           0.96       500
   macro avg       0.89      0.89      0.89       500
weighted avg       0.96      0.96      0.96       500

  Confusion matrix  TP=37  FP=9  FN=9  TN=445

────────────────────────────────────────────────────────────
  LOF
────────────────────────────────────────────────────────────
              precision    recall  f1-score   support

      Normal       0.99      0.99      0.99       454
     Anomaly       0.91      0.91      0.91        46

    accuracy                           0.98       500
   macro avg       0.95      0.95      0.95       500
weighted avg       0.98      0.98      0.98       500

  Confusion matrix  TP=42  FP

One-line summary for each attribute and its exact performance result:

 
* Accuracy (0.96): The model correctly classifies 96% of all samples, showing outstanding overall performance.
* Normal F1-Score (0.98): The model has near-perfect accuracy and balance when identifying normal data points.
* Anomaly F1-Score (0.80): The model shows a highly robust balance between catching anomalies and avoiding false alarms.
* Anomaly Precision (0.80): When the model triggers an alert, it is correct 80% of the time (only 9 false alarms).
* Anomaly Recall (0.80): The model successfully catches 80% of all actual anomalies present in the data.
* True Negatives (445): The model correctly clears 445 normal samples, proving it does not blindly flag everything as an anomaly.
* True Positives (37): The model successfully flags 37 real anomalies, providing highly actionable alerts.
* False Negatives (9): The model completely misses 9 real anomalies, letting them slip through as normal data.
* False Positives (9): The model creates 9 false alarms, incorrectly flagging normal samples as anomalies. [1, 2, 3, 4] 
 

> Tip : adjust the model's parameters to reduce either the missed anomalies or the false alarms 
 

Random Cut Forest (RCF) is an evolution of Isolation Forest (IF) specifically designed by Amazon to handle streaming data and high-dimensional noise. While both isolate anomalies using trees, they differ fundamentally in how they split data, handle updates, and calculate anomaly scores.   
### Core Component Comparison

| Feature | Isolation Forest (IF) | Random Cut Forest (RCF) |
|---|---|---|
| How Dimensions are Cut | Picks a random attribute completely uniformly. | Probability of choosing an attribute depends on its range size. |
| Streaming & Real-Time | Must retrain the whole model on new batch data. | Designed to update dynamically as stream data rolls in. |
| Scoring Basis | Path depth: how early a point gets isolated. | Displacement: how much a point alters the tree structure. |
| Clustered Anomalies | Struggles when anomalies bunch together. | Excellent at spotting anomalies that mask each other. |
| Availability on AWS | Run manually via custom code or containers. | Built-in native algorithm across core AWS services. |

------------------------------
##  Differences
### 1. Splitting Strategy (The "Cuts")

 
* Isolation Forest: Picks a feature completely at random, then splits randomly between the minimum and maximum value. This means an irrelevant feature with tiny variance can get chosen just as often as an important feature.  
* Random Cut Forest: The probability of picking a feature is proportional to its range (bounding box size). Features with massive variance or sudden spikes are automatically prioritized for splits, isolating anomalies much faster. 
 

### 2. Scoring Mechanism

 
* Isolation Forest: Uses tree depth. Anomalies are closer to the root, normal data is deep down.
* Random Cut Forest: Uses collusive displacement. Instead of just measuring depth, it calculates how much the tree structure shifts or expands if you insert or remove that data point. 
 

### 3. Streaming and Time-Series Data

 
* Isolation Forest: Static algorithm. If your data trends change, you must re-collect the dataset and retrain a new model from scratch.
* Random Cut Forest: Dynamic algorithm. It supports a sliding-window reservoir sampling technique. As new data streams in, old points evaporate from the trees and new ones are inserted on-the-fly without rebuilding the forest.
 

### 4. Managing Group Anomalies

 
* Isolation Forest: If multiple anomalies cluster close to one another, they hide each other. The algorithm requires many cuts to separate them, making them look like normal deep-nested data.  
* Random Cut Forest: Because it measures global displacement, it recognizes if a group of points is causing a major tree distortion, easily flagging coordinated or repeating anomalies.  
 

### 5. Ecosystem & Implementation
 
* Isolation Forest: The industry standard for local data science via scikit-learn. On AWS, you run it inside an Amazon SageMaker AI container or [AWS Glue notebook](https://docs.aws.amazon.com/glue/latest/dg/using-notebooks-overview.html) using standard Python libraries.
* Random Cut Forest: Optimized specifically as a first-class proprietary tool by AWS. It is natively baked into services like Amazon Kinesis Data Analytics for live stream processing and [SageMaker RCF](https://docs.aws.amazon.com/sagemaker/latest/dg/randomcutforest.html) for out-of-the-box training.  
 
 